In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

#
# Change "Questionnnair" -> "Questionnaire"
# if your actual folder uses the correct spelling.
#
DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "WHQ_I.xpt"
csv_file = DATA_DIR / "WHQ_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 WHQ_I XPT -> CSV")
print("=" * 100)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WHD010",
    "WHD020",
    "WHQ030",
    "WHQ040",
    "WHD050",
    "WHQ060",
    "WHQ070",
    "WHD080A",
    "WHD080B",
    "WHD080C",
    "WHD080D",
    "WHD080E",
    "WHD080F",
    "WHD080G",
    "WHD080H",
    "WHD080I",
    "WHD080J",
    "WHD080K",
    "WHD080M",
    "WHD080N",
    "WHD080O",
    "WHD080P",
    "WHD080Q",
    "WHD080R",
    "WHD080S",
    "WHD080T",
    "WHD080U",
    "WHD080L",
    "WHQ225",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150",
    "WHQ190",
    "WHQ200"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "WHQ_I columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 6327:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 37:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,327 rows x 37 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Direct inspection of attached WHQ_I.xpt:
#
#     string variables              = NONE
#     variables with real fractions = NONE
#
#
# Weight / height variables are released here in
# whole pounds / inches.
#
# Examples:
#
#     WHD010 = height in inches
#     WHD020 = current weight in pounds
#     WHD050 = weight one year ago
#     WHD110 = weight ten years ago
#     WHD120 = weight at age 25
#     WHD130 = height at age 25
#     WHD140 = greatest weight
#
#
# We do NOT round any values.
#
# We first prove that every populated value is
# mathematically whole-number and only then convert
# to nullable Int64.
#
#
# IMPORTANT:
#
# This decision applies to THIS WHQ_I release only.
#
# Your broader rule remains:
#
#     genuine Dietary decimals
#     genuine Lab decimals
#     genuine Examination decimals
#     survey/sample weights
#
# remain exactly AS IS.
#

DECIMAL_COLUMNS = []

STRING_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


if len(INTEGER_COLUMNS) != 37:

    raise ValueError(
        "Expected exactly 37 whole-number fields."
    )


print(
    "\n--- FIELD TYPE PLAN ---"
)

print(
    "Character fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN SAS XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation of zero:
#
#     5.397605346934028e-79
#
#
# IMPORTANT:
#
# Attached WHQ_I contains ZERO occurrences.
#
# Therefore NO tiny->zero replacement is performed.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 37 WHQ_I variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0)
    .sum()
)


ordinary_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


ordinary_zero_total = int(
    ordinary_zero_counts.sum()
)


print(
    "\n--- ORIGINAL XPT ZERO CHECK ---"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Ordinary numeric zeros:",
    f"{ordinary_zero_total:,}"
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in WHQ_I."
    )


if ordinary_zero_total != 0:

    print(
        ordinary_zero_counts[
            ordinary_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected ordinary zero values "
        "found in WHQ_I."
    )


print(
    "PASS: WHQ_I contains 0 tiny artifacts "
    "and 0 ordinary numeric zeros."
)


# ============================================================
# 6. NO ZERO REPAIR REQUIRED
# ============================================================

#
# Since WHQ_I contains no zero-decoding problem,
# the untouched XPT is itself the exact numerical
# fidelity reference.
#

expected_df = source_df.copy()


print(
    "\nPASS: No tiny-to-zero repair required."
)


# ============================================================
# 7. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# This is done BEFORE Int64 conversion.
#
# np.round() is used ONLY for diagnostic comparison.
#
# It does NOT overwrite values.
#

print(
    "\n--- FRACTIONAL VALUE DETECTION ---"
)


fractional_counts = {}

fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        fractional_counts[col] = 0
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional_mask.sum()
    )


    fractional_counts[col] = count


    if count > 0:

        fractional_columns.append(
            col
        )


print(
    "Variables containing fractional observations:",
    fractional_columns
)


if fractional_columns:

    raise ValueError(
        "STOP: Unexpected genuine decimal values found.\n\n"
        "These values will NOT be rounded:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: No fractional observations "
    "found in any of the 37 variables."
)


# ============================================================
# 8. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print(
    "\n--- SEQN CHECK ---"
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 83732:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 9. PROVE ALL 37 FIELDS ARE WHOLE-NUMBER
# ============================================================

whole_number_failures = []


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    whole_mask = np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if not whole_mask.all():

        whole_number_failures.append(
            col
        )


if whole_number_failures:

    raise ValueError(
        "Unexpected fractional values in:\n"
        + str(
            whole_number_failures
        )
    )


print(
    "PASS: All 37 WHQ_I variables are "
    "mathematically whole-number."
)


# ============================================================
# 10. CONVERT ALL 37 FIELDS TO NULLABLE Int64
# ============================================================

#
# CRITICAL:
#
# NO .round() is used during conversion.
#
# Values such as:
#
#     150.0 -> 150
#
# are numerically identical.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All 37 fields converted to "
    "nullable Int64 without rounding."
)


# ============================================================
# 11. HELPER - EXACT FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequency distribution differs.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. HELPER - RANGE / SPECIAL CODE CHECK
# ============================================================

def check_range_special(
    column,
    minimum,
    maximum,
    expected_range_count,
    expected_special,
    expected_missing
):

    range_count = int(
        df[column]
        .between(
            minimum,
            maximum
        )
        .sum()
    )


    actual_special = {
        code: int(
            df[column]
            .eq(code)
            .sum()
        )
        for code in expected_special
    }


    missing = int(
        df[column]
        .isna()
        .sum()
    )


    print(
        f"\n--- {column} RANGE CHECK ---"
    )

    print(
        f"{minimum}-{maximum}:",
        f"{range_count:,}"
    )

    print(
        "Special codes:",
        actual_special
    )

    print(
        "Missing:",
        f"{missing:,}"
    )


    if range_count != expected_range_count:

        raise ValueError(
            f"{column} valid-range count differs."
        )


    if actual_special != expected_special:

        raise ValueError(
            f"{column} special-code distribution differs."
        )


    if missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs."
        )


    print(
        f"PASS: {column} range/special codes validated."
    )


# ============================================================
# 13. CDC CHECK - WHD010
# ============================================================

#
# Current self-reported height, inches:
#
#     47-84 = 6,157
#     7777  = 1
#     9999  = 167
#     Missing = 2
#

check_range_special(
    "WHD010",
    47,
    84,
    6157,
    {
        7777: 1,
        9999: 167
    },
    2
)


# ============================================================
# 14. CDC CHECK - WHD020
# ============================================================

#
# Current self-reported weight, pounds.
#

check_range_special(
    "WHD020",
    75,
    600,
    6211,
    {
        7777: 9,
        9999: 107
    },
    0
)


# ============================================================
# 15. CDC CHECK - WHQ030
# ============================================================

check_frequency(
    "WHQ030",
    {
        1: 3047,
        2: 391,
        3: 2872,
        7: 1,
        9: 16
    },
    0
)


# ============================================================
# 16. CDC CHECK - WHQ040
# ============================================================

check_frequency(
    "WHQ040",
    {
        1: 611,
        2: 3790,
        3: 1922,
        7: 1,
        9: 3
    },
    0
)


# ============================================================
# 17. CDC CHECK - WHD050
# ============================================================

#
# Self-reported weight one year ago.
#

check_range_special(
    "WHD050",
    60,
    600,
    6185,
    {
        7777: 4,
        9999: 138
    },
    0
)


# ============================================================
# 18. CDC CHECK - WHQ060
# ============================================================

check_frequency(
    "WHQ060",
    {
        1: 802,
        2: 465,
        9: 2
    },
    5058
)


# ============================================================
# 19. CDC CHECK - WHQ070
# ============================================================

check_frequency(
    "WHQ070",
    {
        1: 2116,
        2: 3392,
        7: 1,
        9: 3
    },
    815
)


# ============================================================
# 20. MULTI-RESPONSE WEIGHT-LOSS METHODS
# ============================================================

#
# WHD080A-U / L are "code all that apply" variables.
#
# A populated variable contains only its designated
# category code.
#

EXPECTED_METHOD_FIELDS = {

    "WHD080A": {
        "code": 10,
        "count": 1756
    },

    "WHD080B": {
        "code": 11,
        "count": 940
    },

    "WHD080C": {
        "code": 12,
        "count": 945
    },

    "WHD080D": {
        "code": 13,
        "count": 1888
    },

    "WHD080E": {
        "code": 14,
        "count": 570
    },

    "WHD080F": {
        "code": 15,
        "count": 266
    },

    "WHD080G": {
        "code": 16,
        "count": 150
    },

    "WHD080H": {
        "code": 17,
        "count": 99
    },

    "WHD080I": {
        "code": 31,
        "count": 93
    },

    "WHD080J": {
        "code": 32,
        "count": 196
    },

    "WHD080K": {
        "code": 33,
        "count": 37
    },

    "WHD080M": {
        "code": 34,
        "count": 1579
    },

    "WHD080N": {
        "code": 30,
        "count": 150
    },

    "WHD080O": {
        "code": 41,
        "count": 818
    },

    "WHD080P": {
        "code": 42,
        "count": 44
    },

    "WHD080Q": {
        "code": 43,
        "count": 1638
    },

    "WHD080R": {
        "code": 44,
        "count": 1177
    },

    "WHD080S": {
        "code": 45,
        "count": 1178
    },

    "WHD080T": {
        "code": 46,
        "count": 1335
    },

    "WHD080U": {
        "code": 35,
        "count": 16
    },

    "WHD080L": {
        "code": 40,
        "count": 30
    }
}


print(
    "\n--- WEIGHT LOSS METHOD FIELDS ---"
)


for col, expected in EXPECTED_METHOD_FIELDS.items():

    nonmissing = (
        df[col]
        .dropna()
    )


    actual_count = int(
        nonmissing.shape[0]
    )


    observed_codes = set(
        nonmissing
        .astype(int)
        .unique()
        .tolist()
    )


    expected_code = (
        expected["code"]
    )


    expected_count = (
        expected["count"]
    )


    print(
        f"{col}: "
        f"code={expected_code}, "
        f"count={actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} records, "
            f"found {actual_count:,}."
        )


    if observed_codes != {expected_code}:

        raise ValueError(
            f"{col}: unexpected codes "
            f"{observed_codes}."
        )


print(
    "PASS: All 21 weight-loss method "
    "variables match CDC."
)


# ============================================================
# 21. CDC CHECK - WHQ225
# ============================================================

check_frequency(
    "WHQ225",
    {
        1: 1772,
        2: 908,
        3: 296,
        4: 220,
        5: 3083,
        7: 2,
        9: 46
    },
    0
)


# ============================================================
# 22. CDC CHECK - WHD110
# ============================================================

#
# Weight ten years ago.
#
# Asked for age >= 36.
#

check_range_special(
    "WHD110",
    80,
    700,
    3991,
    {
        7777: 2,
        9999: 144
    },
    2190
)


# ============================================================
# 23. CDC CHECK - WHD120
# ============================================================

#
# Weight at age 25.
#
# Asked for age >= 27.
#

check_range_special(
    "WHD120",
    70,
    420,
    4817,
    {
        7777: 3,
        9999: 237
    },
    1270
)


# ============================================================
# 24. CDC CHECK - WHD130
# ============================================================

#
# Height at age 25.
#
# Asked for age >= 50.
#

check_range_special(
    "WHD130",
    51,
    84,
    2670,
    {
        7777: 2,
        9999: 148
    },
    3507
)


# ============================================================
# 25. CDC CHECK - WHD140
# ============================================================

#
# Greatest self-reported weight.
#

check_range_special(
    "WHD140",
    75,
    559,
    5883,
    {
        7777: 8,
        9999: 101
    },
    335
)


# ============================================================
# 26. CDC CHECK - WHQ150
# ============================================================

#
# Age when participant weighed the most.
#
# CDC disclosure editing:
#
#     80 = age 80 years or older
#

whq150_range = int(
    df["WHQ150"]
    .between(
        10,
        79
    )
    .sum()
)


whq150_80 = int(
    df["WHQ150"]
    .eq(80)
    .sum()
)


whq150_77777 = int(
    df["WHQ150"]
    .eq(77777)
    .sum()
)


whq150_99999 = int(
    df["WHQ150"]
    .eq(99999)
    .sum()
)


whq150_missing = int(
    df["WHQ150"]
    .isna()
    .sum()
)


print(
    "\n--- WHQ150 ---"
)

print(
    "Age 10-79:",
    f"{whq150_range:,}"
)

print(
    "Age 80+:",
    whq150_80
)

print(
    "77777:",
    whq150_77777
)

print(
    "99999:",
    whq150_99999
)

print(
    "Missing:",
    whq150_missing
)


if whq150_range != 5736:

    raise ValueError(
        "WHQ150 age 10-79 count differs."
    )


if whq150_80 != 95:

    raise ValueError(
        "WHQ150 age-80+ count differs."
    )


if whq150_77777 != 0:

    raise ValueError(
        "Unexpected WHQ150 refused values."
    )


if whq150_99999 != 46:

    raise ValueError(
        "WHQ150 don't-know count differs."
    )


if whq150_missing != 450:

    raise ValueError(
        "WHQ150 missing count differs."
    )


print(
    "PASS: WHQ150 matches CDC."
)


# ============================================================
# 27. CDC CHECK - WHQ190
# ============================================================

check_frequency(
    "WHQ190",
    {
        1: 58,
        2: 6268,
        9: 1
    },
    0
)


# ============================================================
# 28. CDC CHECK - WHQ200
# ============================================================

#
# Age when participant had most recent
# weight-loss surgery.
#
# Populated exactly for WHQ190=1.
#

whq200_range = int(
    df["WHQ200"]
    .between(
        17,
        74
    )
    .sum()
)


whq200_missing = int(
    df["WHQ200"]
    .isna()
    .sum()
)


print(
    "\n--- WHQ200 ---"
)

print(
    "Age 17-74:",
    whq200_range
)

print(
    "Missing:",
    whq200_missing
)


if whq200_range != 58:

    raise ValueError(
        "WHQ200 age range count differs."
    )


if int(
    df["WHQ200"]
    .eq(77777)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected WHQ200 refused values."
    )


if int(
    df["WHQ200"]
    .eq(99999)
    .sum()
) != 0:

    raise ValueError(
        "Unexpected WHQ200 don't-know values."
    )


if whq200_missing != 6269:

    raise ValueError(
        "WHQ200 missing count differs."
    )


print(
    "PASS: WHQ200 matches CDC."
)


# ============================================================
# 29. WHQ190 -> WHQ200 ROUTING CHECK
# ============================================================

whq200_expected = (
    df["WHQ190"]
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


whq200_present = (
    df["WHQ200"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


whq200_routing_mismatch = int(
    np.sum(
        whq200_expected
        !=
        whq200_present
    )
)


print(
    "\n--- WHQ190 / WHQ200 ROUTING ---"
)

print(
    "WHQ190=1:",
    int(
        whq200_expected.sum()
    )
)

print(
    "WHQ200 populated:",
    int(
        whq200_present.sum()
    )
)

print(
    "Routing mismatches:",
    whq200_routing_mismatch
)


if int(
    whq200_expected.sum()
) != 58:

    raise ValueError(
        "Unexpected WHQ200 eligibility count."
    )


if whq200_routing_mismatch != 0:

    raise ValueError(
        "WHQ200 availability differs "
        "from WHQ190=1."
    )


print(
    "PASS: WHQ190/WHQ200 routing validated."
)


# ============================================================
# 30. FINAL ZERO CHECK BEFORE EXPORT
# ============================================================

#
# Original XPT contained:
#
#     tiny artifacts = 0
#     ordinary zeros = 0
#
#
# Therefore no real numeric zeros should appear
# after processing either.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print(
    "\n--- FINAL ZERO CHECK BEFORE EXPORT ---"
)

print(
    final_zero_counts
)


if final_zero_counts:

    raise ValueError(
        "Unexpected numeric zeros found:\n"
        + str(
            final_zero_counts
        )
    )


print(
    "PASS: No numeric zeros created."
)


# ============================================================
# 31. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# WHQ_I required NO numerical corrections.
#
# Therefore every value must remain numerically
# identical to the original XPT.
#

print(
    "\n--- VALUE FIDELITY BEFORE EXPORT ---"
)


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: All 37 variables remain numerically "
    "identical to source XPT."
)


# ============================================================
# 32. STORE MISSING COUNTS BEFORE EXPORT
# ============================================================

expected_missing_counts = {
    col: int(
        expected_df[col]
        .isna()
        .sum()
    )
    for col in EXPECTED_COLUMNS
}


# ============================================================
# 33. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
#
# WHQ_I has no genuine fractional values.
#
# For other NHANES files containing genuine decimals,
# continue preserving those decimal variables as float.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 34. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 35. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(
            csv_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 37 WHQ_I variables."
)


# ============================================================
# 36. MISSING-VALUE FIDELITY
# ============================================================

missing_position_mismatches = 0

missing_count_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_position_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


    actual_missing_count = int(
        roundtrip_df[col]
        .isna()
        .sum()
    )


    if (
        actual_missing_count
        !=
        expected_missing_counts[col]
    ):

        missing_count_failures.append(
            col
        )


print(
    "\n--- MISSING VALUE FIDELITY ---"
)

print(
    "Missing-position mismatches:",
    missing_position_mismatches
)


if missing_position_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


if missing_count_failures:

    raise ValueError(
        "Missing counts changed in:\n"
        + str(
            missing_count_failures
        )
    )


print(
    "PASS: Missing values preserved exactly "
    "across all 37 variables."
)


# ============================================================
# 37. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 38. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print(
    "\n--- FINAL TINY VALUE CHECK ---"
)

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 39. RAW CSV .0 CHECK
# ============================================================

#
# WHQ_I contains no genuine fractional values.
#
# Therefore checking all 37 fields for unwanted ".0"
# is safe for THIS specific file.
#
#
# Examples:
#
#     SEQN:
#         83732.0 -> 83732
#
#     WHD020:
#         150.0 -> 150
#
#     WHD050:
#         165.0 -> 165
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- RAW CSV .0 CHECK ---"
)

print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' "
    "in any WHQ_I field."
)


# ============================================================
# 40. FINAL ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    count = int(
        values
        .eq(0)
        .sum()
    )


    if count > 0:

        csv_zero_counts[col] = count


print(
    "\n--- FINAL CSV ZERO CHECK ---"
)

print(
    csv_zero_counts
)


if csv_zero_counts:

    raise ValueError(
        "Unexpected zero values found in final CSV:\n"
        + str(
            csv_zero_counts
        )
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 41. FINAL WHQ190 / WHQ200 ROUTING
# ============================================================

final_whq190 = pd.to_numeric(
    raw_csv_df["WHQ190"],
    errors="coerce"
)


expected_whq200_present = (
    final_whq190
    .eq(1)
    .fillna(False)
    .to_numpy(
        dtype=bool
    )
)


actual_whq200_present = (
    raw_csv_df["WHQ200"]
    .ne("")
    .to_numpy(
        dtype=bool
    )
)


final_routing_mismatch = int(
    np.sum(
        expected_whq200_present
        !=
        actual_whq200_present
    )
)


print(
    "\n--- FINAL WHQ190 / WHQ200 ROUTING ---"
)

print(
    "Routing mismatches:",
    final_routing_mismatch
)


if final_routing_mismatch != 0:

    raise ValueError(
        "Final WHQ190/WHQ200 routing changed."
    )


print(
    "PASS: Final surgery routing preserved."
)


# ============================================================
# 42. FINAL KEY RANGE CHECKS
# ============================================================

final_numeric = {}


for col in [
    "WHD010",
    "WHD020",
    "WHD050",
    "WHD110",
    "WHD120",
    "WHD130",
    "WHD140",
    "WHQ150",
    "WHQ200"
]:

    final_numeric[col] = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


FINAL_RANGE_EXPECTATIONS = {

    "WHD010": {
        "min": 47,
        "max": 84,
        "count": 6157
    },

    "WHD020": {
        "min": 75,
        "max": 600,
        "count": 6211
    },

    "WHD050": {
        "min": 60,
        "max": 600,
        "count": 6185
    },

    "WHD110": {
        "min": 80,
        "max": 700,
        "count": 3991
    },

    "WHD120": {
        "min": 70,
        "max": 420,
        "count": 4817
    },

    "WHD130": {
        "min": 51,
        "max": 84,
        "count": 2670
    },

    "WHD140": {
        "min": 75,
        "max": 559,
        "count": 5883
    },

    "WHQ150": {
        "min": 10,
        "max": 79,
        "count": 5736
    },

    "WHQ200": {
        "min": 17,
        "max": 74,
        "count": 58
    }
}


print(
    "\n--- FINAL KEY RANGE CHECKS ---"
)


for col, expected in FINAL_RANGE_EXPECTATIONS.items():

    count = int(
        final_numeric[col]
        .between(
            expected["min"],
            expected["max"]
        )
        .sum()
    )


    print(
        f"{col}:",
        count
    )


    if count != expected["count"]:

        raise ValueError(
            f"Final {col} range count changed."
        )


print(
    "PASS: Final measurement/range counts preserved."
)


# ============================================================
# 43. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    scientific_count += int(
        raw_csv_df[col]
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\n--- SCIENTIFIC NOTATION CHECK ---"
)

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 44. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final SEQN still contains '.0'."
    )


print(
    "PASS: Final SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 45. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "WHQ_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)


print(
    "Cycle: 2015-2016"
)


print(
    "Component: Questionnaire - Weight History"
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)


print(
    "Original tiny XPORT artifacts:",
    tiny_total
)


print(
    "Original ordinary numeric zeros:",
    ordinary_zero_total
)


print(
    "Validated zero restorations:",
    "NONE"
)


print(
    "Variables with genuine fractional values:",
    fractional_columns
)


print(
    "Whole-number released fields:",
    len(INTEGER_COLUMNS)
)


print(
    "Current height records:",
    f"{int(df['WHD010'].notna().sum()):,}"
)


print(
    "Current weight records:",
    f"{int(df['WHD020'].notna().sum()):,}"
)


print(
    "Weight one year ago records:",
    f"{int(df['WHD050'].notna().sum()):,}"
)


print(
    "Weight ten years ago records:",
    f"{int(df['WHD110'].notna().sum()):,}"
)


print(
    "Weight at age 25 records:",
    f"{int(df['WHD120'].notna().sum()):,}"
)


print(
    "Greatest weight records:",
    f"{int(df['WHD140'].notna().sum()):,}"
)


print(
    "Weight-loss surgery yes:",
    int(
        df["WHQ190"]
        .eq(1)
        .sum()
    )
)


print(
    "WHQ190/WHQ200 routing mismatches:",
    whq200_routing_mismatch
)


print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)


print(
    "Missing-position differences:",
    "0 expected"
)


print(
    "Tiny values remaining:",
    "0 expected"
)


print(
    "Unnecessary '.0':",
    "0 expected"
)


print(
    "Scientific notation:",
    "0 expected"
)


print(
    "\nFINAL RULE:"
)


print(
    "WHQ_I requires NO tiny-zero restoration."
)


print(
    "Every released value in this attachment "
    "is mathematically whole-number."
)


print(
    "All 37 fields can therefore lose unnecessary "
    "'.0' without rounding or changing numeric values."
)


print(
    "No df.round() and no float_format are used."
)


print(
    "Your broader NHANES fidelity rule remains unchanged:"
)


print(
    "Genuine Dietary and other genuine continuous "
    "decimal values remain exactly AS IS."
)

NHANES 2015-2016 WHQ_I XPT -> CSV
Rows: 6,327
Columns: 37
PASS: Structure = 6,327 rows x 37 columns.

--- FIELD TYPE PLAN ---
Character fields: []
Genuine decimal fields: []
Whole-number fields: 37

--- ORIGINAL XPT ZERO CHECK ---
Tiny XPORT artifacts: 0
Variables containing tiny artifacts: 0
Ordinary numeric zeros: 0
PASS: WHQ_I contains 0 tiny artifacts and 0 ordinary numeric zeros.

PASS: No tiny-to-zero repair required.

--- FRACTIONAL VALUE DETECTION ---
Variables containing fractional observations: []
PASS: No fractional observations found in any of the 37 variables.

--- SEQN CHECK ---
Minimum: 83732
Maximum: 93702
Missing: 0
Duplicates: 0
PASS: SEQN 83732-93702 validated.
PASS: All 37 WHQ_I variables are mathematically whole-number.
PASS: All 37 fields converted to nullable Int64 without rounding.

--- WHD010 RANGE CHECK ---
47-84: 6,157
Special codes: {7777: 1, 9999: 167}
Missing: 2
PASS: WHD010 range/special codes validated.

--- WHD020 RANGE CHECK ---
75-600: 6,211
Special c